In [2]:
## Für die Nutzung des Modells Llama

from openai import OpenAI

client = OpenAI(base_url="http://localhost:8002/v1", api_key="none")



In [ ]:
#im Terminal: um zu gucken wieviele Tokens erlaubt das Modell pro Anfrage
#curl http://localhost:8002/v1/models | jq

### Bereinigung der last_llm_experiments

In [2]:
import json
import re

input_path = "last_llm_experiments.json"
output_path = "last_clean_llm_experiments.json"

with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
    for line in infile:
        if not line.strip():
            continue
        data = json.loads(line)
        text = data.get("text", "")

        # 🧹 1. "LLM Output Starts here:" entfernen (am Anfang, optional mit Leerzeichen danach)
        text = re.sub(r'^\s*LLM Output Starts here:\s*', '', text)

        # 🧽 2. alle echten oder escaped Zeilenumbrüche entfernen
        text = re.sub(r'\s*\n+\s*', ' ', text)
        text = text.replace("\\n", " ")

        # 🧼 3. überflüssige Leerzeichen am Anfang/Ende weg
        data["text"] = text.strip()

        # 💾 4. wieder als JSON-Zeile speichern
        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")


### Embedding erzeugung mit Llama für die Experimente

In [3]:
import pandas as pd
from openai import OpenAI
import numpy as np
from tqdm import tqdm  # Fortschrittsanzeige

# --- 1️⃣ Daten einlesen ---
df = pd.read_json("last_clean_llm_experiments.json", lines=True)


# --- 3️⃣ Neue Spalte vorbereiten ---
col_name = "embedding_Llama-3.1_8B-Instruct"
df[col_name] = None

# --- 4️⃣ Embeddings erzeugen (Batchweise optional) ---
embeddings = []

for text in tqdm(df["text"], desc="Berechne Embeddings"):
    if not isinstance(text, str) or not text.strip():
        embeddings.append(None)
        continue
    
    response = client.embeddings.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        input=text
    )
    emb = response.data[0].embedding
    embeddings.append(emb)

df[col_name] = embeddings

# --- 5️⃣ Datei speichern ---
df.to_parquet("last_clean_llm_experiments_with_embeddings.parquet")

print(f"✅ Fertig! {len(df)} Texte mit Spalte '{col_name}' gespeichert.")


Berechne Embeddings: 100%|██████████| 442/442 [01:31<00:00,  4.85it/s]


✅ Fertig! 442 Texte mit Spalte 'embedding_Llama-3.1_8B-Instruct' gespeichert.


In [4]:
# Prüfen ob das lokale Llama automatisch normalisiert.!


response = client.embeddings.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    input="fghztu"
)

import numpy as np
vec = np.array(response.data[0].embedding)
print("Norm =", np.linalg.norm(vec))


Norm = 1.0000004229005754


In [1]:
import pandas as pd

df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")



In [2]:
df

,experiment_id,audio_id,model_name,run_id,type,tags,text,embedding_Llama-3.1_8B-Instruct,embedding_Jina_v3,BAAI/bge-m3,Alibaba-NLP/gte-multilingual-base,aari1995/German_Semantic_V3b
0,1765,Anam_Beinschmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_03,output,"[AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...","Patient: Frau Mayerhose, 55 Jahre alt, geboren...","[-0.0008558990084566176, -0.020405177026987076...","[0.05810546875, -0.08203125, 0.0225830078125, ...","[0.06356414407491684, -0.01877567172050476, -0...","[-0.05058562010526657, 0.10307077318429947, -0...","[0.046458642929792404, -0.017108825966715813, ..."
1,1762,Anam_Beinschmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_01,output,"[AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...","Patient: Frau Mayerhose Katarina, 55 Jahre alt...","[-0.001057965331710875, -0.01998530514538288, ...","[0.06689453125, -0.11328125, 0.002471923828125...","[0.05369797721505165, -0.006991442758589983, -...","[-0.05944622680544853, 0.1052369698882103, 0.0...","[0.059302035719156265, -0.02372737042605877, 0..."
2,1767,Anam_Beinschmerzen,MedGemma 27B Instruct,P1_01_01,output,"[AI, LLM, STT, Summary, MedGemma 27B Instruct,...","Okay, hier ist die Zusammenfassung des Gespräc...","[0.007169963791966438, -0.01883453130722046, 0...","[0.10546875, -0.08935546875, 0.02734375, 0.124...","[0.012655723839998245, -0.00448342552408576, -...","[-0.04368281364440918, 0.0966692790389061, -0....","[0.06236836686730385, -0.030042340978980064, 0..."
3,1766,Anam_Beinschmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_04,output,"[AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...","Patient: Frau Mayerhose Katarina, 55 Jahre alt...","[-0.0013266407186165452, -0.01246361993253231,...","[0.06982421875, -0.07568359375, -0.02111816406...","[0.06326344609260559, -0.011052247136831284, -...","[-0.06601697206497192, 0.11261804401874542, 0....","[0.05892357975244522, -0.027058755978941917, 0..."
4,1769,Anam_Beinschmerzen,MedGemma 27B Instruct,P1_01_03,output,"[AI, LLM, STT, Summary, MedGemma 27B Instruct,...","Okay, hier ist die Zusammenfassung des Gespräc...","[0.004689760971814394, -0.029031852260231972, ...","[0.1142578125, -0.08349609375, 0.0284423828125...","[0.02266274020075798, 0.014821775257587433, -0...","[-0.046744443476200104, 0.09816469252109528, 0...","[0.0632222443819046, -0.015532443299889565, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...
437,1819,Anam_Hyperthyreose,Meta Llama 3.1 8B Instruct,P1_01_02,output,"[AI, LLM, STT, Summary, Meta Llama 3.1 8B Inst...","LLM Output Starts here ### Patient: Frau Maya,...","[0.0038856349419802427, -0.009560339152812958,...","[0.07666015625, -0.046630859375, 0.0654296875,...","[-0.04150008037686348, -0.0047930460423231125,...","[-0.051061030477285385, 0.027630450204014778, ...","[0.04619033262133598, 0.0047134580090641975, 0..."
438,1734,Anam_Bauchschmerzen,Meta Llama 3.1 8B Instruct,P1_01_04,output,"[AI, LLM, STT, Summary, Meta Llama 3.1 8B Inst...","### Patient: Frau Milberg, Anke, ist 45 Jahre ...","[-0.009398126974701881, -0.02021690085530281, ...","[0.0147705078125, -0.049072265625, 0.076660156...","[0.020849689841270447, 0.030632486566901207, -...","[-0.10973813384771347, 0.10469478368759155, -0...","[0.024189062416553497, 0.01349211111664772, -0..."
439,1731,Anam_Bauchschmerzen,Meta Llama 3.1 8B Instruct,P1_01_02,output,"[AI, LLM, STT, Summary, Meta Llama 3.1 8B Inst...","### Patient: Frau Milberg, Anke, ist 45 Jahre ...","[-0.003908646292984486, -0.012081270106136799,...","[0.0076904296875, -0.0322265625, 0.05810546875...","[-0.0040303985588252544, 0.030535556375980377,...","[-0.10530617088079453, 0.11809652298688889, 0....","[0.04137205332517624, 0.022197004407644272, -0..."
440,1733,Anam_Bauchschmerzen,Meta Llama 3.1 8B Instruct,P1_01_03,output,"[AI, LLM, STT, Summary, Meta Llama 3.1 8B Inst...","### Patient: Frau Milberg, Anke, ist 45 Jahre ...","[-0.003568394808098674, -0.01866544969379902, ...","[0.0185546875, -0.0247802734375, 0.0595703125,..

## Embedding mit einem anderen Modell jina-embeddings-v3

In [6]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# --- 2️⃣ Modell laden ---
model = SentenceTransformer(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
    device="cuda"
)


/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed.

In [7]:
# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_Jina_v3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=2,         # effizienter auf GPU (mit 20, 10 oder sogar 5 kam eine Fehlermeldung!)
    normalize_embeddings=True #(das habe ich vorher nicht gemaht. Es kann aber wichtig sein!)
)

# --- 5️⃣ Embeddings in DataFrame speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("last_clean_llm_experiments_with_embeddings.parquet")

print(f"✅ {len(df)} Embeddings mit JinaAI v3 berechnet und in '{col_name}' gespeichert.")

Batches: 100%|██████████| 221/221 [00:12<00:00, 17.91it/s]


✅ 442 Embeddings mit JinaAI v3 berechnet und in 'embedding_Jina_v3' gespeichert.


In [1]:
import pandas as pd

# Parquet-Datei laden
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# Kontrolle
print("✅ Datei geladen!")
print(f"Anzahl Zeilen: {len(df)}")
print("Spalten:", list(df.columns))


✅ Datei geladen!
Anzahl Zeilen: 442
Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'type', 'tags', 'text', 'embedding_Llama-3.1_8B-Instruct', 'embedding_Jina_v3', 'BAAI/bge-m3', 'Alibaba-NLP/gte-multilingual-base', 'aari1995/German_Semantic_V3b']


In [2]:
df.iloc[0]


experiment_id                                                                     1765
audio_id                                                            Anam_Beinschmerzen
model_name                                         Llama 3.1 SauerkrautLM 70B Instruct
run_id                                                                        P1_01_03
type                                                                            output
tags                                 [AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...
text                                 Patient: Frau Mayerhose, 55 Jahre alt, geboren...
embedding_Llama-3.1_8B-Instruct      [-0.0008558990084566176, -0.020405177026987076...
embedding_Jina_v3                    [0.05810546875, -0.08203125, 0.0225830078125, ...
BAAI/bge-m3                          [0.06356414407491684, -0.01877567172050476, -0...
Alibaba-NLP/gte-multilingual-base    [-0.05058562010526657, 0.10307077318429947, -0...
aari1995/German_Semantic_V3b         [0.046

In [10]:
df[df["experiment_id"] == 1684]


,experiment_id,audio_id,model_name,run_id,type,tags,text,embedding_Llama-3.1_8B-Instruct,embedding_Jina_v3
238,1684,Anam_Bauchschmerzen,Meta Llama 3.1 8B Instruct,P1_01_01,output,"[AI, LLM, STT, Summary, Meta Llama 3.1 8B Inst...","### Patient: Frau Milberg, Anke, ist 45 Jahre ...","[-0.008937099017202854, -0.018751459196209908,...","[0.020263671875, -0.04052734375, 0.07666015625..."


In [11]:
# Llama 3.1 Embedding-Dimension
len(df["embedding_Llama-3.1_8B-Instruct"].iloc[0])




4096

In [ ]:
# Jina v3 Embedding-Dimension
len(df["embedding_Jina_v3"].iloc[0])

1024

: 

## Embedding mit einem anderen Modell aari1995/German_Semantic_V3b 
### Auf CPU zu langsam und auf GPU keine ausreichende Speicher
#### Dies wurde dann mit nohub gemacht in der Datei aari_nohub.py

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("clean_llm_experiments_with_embeddings.parquet")

# --- 2️⃣ Modell laden ---
model = SentenceTransformer(
    "aari1995/German_Semantic_V3b",
    trust_remote_code=True,
    device="cpu" #bei cuda hat es nicht geklappt CUDA out of memory
)

/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 3️⃣ Neue Spalte definieren ---
col_name = "aari1995/German_Semantic_V3b"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=1,         # effizienter auf GPU (mit 20, 10 oder sogar 5 kam eine Fehlermeldung!)
    normalize_embeddings=True, #(das habe ich vorher nicht gemaht. Es kann aber wichtig sein!)
    #output_value="sentence_embedding_128"
)

# --- 5️⃣ Embeddings in DataFrame speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("last_clean_llm_experiments_with_embeddings.parquet")

print(f"✅ {len(df)} Embeddings mit aari1995/German_Semantic_V3b berechnet und in '{col_name}' gespeichert.")

Batches:   0%|          | 0/442 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 79.11 GiB of which 1.67 GiB is free. Process 3912 has 766.00 MiB memory in use. Process 3039271 has 37.48 GiB memory in use. Process 3041802 has 21.19 GiB memory in use. Including non-PyTorch memory, this process has 18.00 GiB memory in use. Of the allocated memory 13.44 GiB is allocated by PyTorch, and 3.98 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

: 

# BAAI/bge-m3

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# --- 2️⃣ Modell laden ---
model = SentenceTransformer(
    "BAAI/bge-m3",
    trust_remote_code=True,
    device="cuda" #bei cuda hat es nicht geklappt CUDA out of memory
)

/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 3️⃣ Neue Spalte definieren ---
col_name = "BAAI/bge-m3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=1,         # effizienter auf GPU (mit 20, 10 oder sogar 5 kam eine Fehlermeldung!)
    normalize_embeddings=True, #(das habe ich vorher nicht gemaht. Es kann aber wichtig sein!)
)

# --- 5️⃣ Embeddings in DataFrame speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("last_clean_llm_experiments_with_embeddings.parquet")

print(f"✅ {len(df)} Embeddings mit BAAI/bge-m3 berechnet und in '{col_name}' gespeichert.")

Batches: 100%|██████████| 442/442 [00:21<00:00, 20.66it/s]


✅ 442 Embeddings mit BAAI/bge-m3 berechnet und in 'BAAI/bge-m3' gespeichert.


In [2]:
import pandas as pd

# Parquet-Datei laden
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# Kontrolle
print("✅ Datei geladen!")
print(f"Anzahl Zeilen: {len(df)}")
print("Spalten:", list(df.columns))


✅ Datei geladen!
Anzahl Zeilen: 442
Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'type', 'tags', 'text', 'embedding_Llama-3.1_8B-Instruct', 'embedding_Jina_v3', 'BAAI/bge-m3', 'Alibaba-NLP/gte-multilingual-base']


In [9]:
df.iloc[0]

experiment_id                                                                     1765
audio_id                                                            Anam_Beinschmerzen
model_name                                         Llama 3.1 SauerkrautLM 70B Instruct
run_id                                                                        P1_01_03
type                                                                            output
tags                                 [AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...
text                                 Patient: Frau Mayerhose, 55 Jahre alt, geboren...
embedding_Llama-3.1_8B-Instruct      [-0.0008558990084566176, -0.020405177026987076...
embedding_Jina_v3                    [0.05810546875, -0.08203125, 0.0225830078125, ...
BAAI/bge-m3                          [0.06356414407491684, -0.01877567172050476, -0...
Alibaba-NLP/gte-multilingual-base    [-0.05058562010526657, 0.10307077318429947, -0...
aari1995/German_Semantic_V3b         [0.046

# Alibaba-NLP/gte-multilingual-base

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# --- 2️⃣ Modell laden ---
model = SentenceTransformer(
    "Alibaba-NLP/gte-multilingual-base",
    trust_remote_code=True,
    device="cuda" #bei cuda hat es nicht geklappt CUDA out of memory
 
)

/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [2]:
# --- 3️⃣ Neue Spalte definieren ---
col_name = "Alibaba-NLP/gte-multilingual-base"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=1,         # effizienter auf GPU (mit 20, 10 oder sogar 5 kam eine Fehlermeldung!)
    normalize_embeddings=True, #(das habe ich vorher nicht gemaht. Es kann aber wichtig sein!)
)

# --- 5️⃣ Embeddings in DataFrame speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("last_clean_llm_experiments_with_embeddings.parquet")

print(f"✅ {len(df)} Embeddings mit Alibaba-NLP/gte-multilingual-base berechnet und in '{col_name}' gespeichert.")

Batches: 100%|██████████| 442/442 [00:09<00:00, 45.18it/s] 


✅ 442 Embeddings mit Alibaba-NLP/gte-multilingual-base berechnet und in 'Alibaba-NLP/gte-multilingual-base' gespeichert.


In [6]:
import pandas as pd

# Parquet-Datei laden
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# Kontrolle

print("✅ Datei geladen!")
print(f"Anzahl Zeilen: {len(df)}")
print("Spalten:", list(df.columns))


✅ Datei geladen!
Anzahl Zeilen: 442
Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'type', 'tags', 'text', 'embedding_Llama-3.1_8B-Instruct', 'embedding_Jina_v3', 'BAAI/bge-m3', 'Alibaba-NLP/gte-multilingual-base']


In [7]:
df.iloc[0]

experiment_id                                                                     1765
audio_id                                                            Anam_Beinschmerzen
model_name                                         Llama 3.1 SauerkrautLM 70B Instruct
run_id                                                                        P1_01_03
type                                                                            output
tags                                 [AI, LLM, STT, Summary, Llama 3.1 SauerkrautLM...
text                                 Patient: Frau Mayerhose, 55 Jahre alt, geboren...
embedding_Llama-3.1_8B-Instruct      [-0.0008558990084566176, -0.020405177026987076...
embedding_Jina_v3                    [0.05810546875, -0.08203125, 0.0225830078125, ...
BAAI/bge-m3                          [0.06356414407491684, -0.01877567172050476, -0...
Alibaba-NLP/gte-multilingual-base    [-0.05058562010526657, 0.10307077318429947, -0...
aari1995/German_Semantic_V3b         [0.046

## Ab hier ist für die Kosinus Ähnlichkeiten Berechnung 

In [8]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

emb_col = "embedding_Jina_v3"
results = []

for audio in df["audio_id"].unique():
    subset = df[df["audio_id"] == audio]
    ref = subset[subset["model_name"] == "OpenAI GPT-5"]
    if len(ref) != 1:
        continue  # skip if keine eindeutige Referenz
    
    ref_emb = np.array(ref.iloc[0][emb_col]).reshape(1, -1)
    others = subset[subset["model_name"] != "OpenAI GPT-5"]

    for _, row in others.iterrows():
        sim = cosine_similarity(ref_emb, np.array(row[emb_col]).reshape(1, -1))[0, 0]
        results.append({
            "audio_id": audio,
            "model_name": row["model_name"],
            "run_id": row["run_id"],
            "similarity_to_ref": sim
        })
        
import pandas as pd
res_df = pd.DataFrame(results)
model_means = res_df.groupby("model_name")["similarity_to_ref"].mean().sort_values(ascending=False)
print(model_means)


model_name
OpenAI GPT OSS 120B                    0.932444
MedGemma 27B Instruct                  0.897478
Llama 3.1 SauerkrautLM 70B Instruct    0.889517
Meta Llama 3.1 8B Instruct             0.870434
Name: similarity_to_ref, dtype: float64


In [9]:
res_df

,audio_id,model_name,run_id,similarity_to_ref
0,Anam_Thorakale Schmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_04,0.919486
1,Anam_Thorakale Schmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_03,0.911437
2,Anam_Thorakale Schmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_02,0.930142
3,Anam_Thorakale Schmerzen,Llama 3.1 SauerkrautLM 70B Instruct,P1_01_01,0.921986
4,Anam_Thorakale Schmerzen,OpenAI GPT OSS 120B,P1_01_04,0.917343
...,...,...,...,...
411,Anam_Fahrradunfall_02,OpenAI GPT OSS 120B,P1_01_02,0.919838
412,Anam_Fahrradunfall_02,OpenAI GPT OSS 120B,P1_01_03,0.938196
413,Anam_Fahrradunfall_02,Meta Llama 3.1 8B Instruct,P1_01_01,0.793270
414,Anam_Fahrradunfall_02,Meta Llama 3.1 8B Instruct,P1_01_02,0.862675


In [5]:
results

[{'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'OpenAI GPT OSS 120B',
  'run_id': 'P1_01_04',
  'similarity_to_ref': np.float64(0.9260884296886681)},
 {'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'OpenAI GPT OSS 120B',
  'run_id': 'P1_01_03',
  'similarity_to_ref': np.float64(0.931258829003212)},
 {'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'OpenAI GPT OSS 120B',
  'run_id': 'P1_01_02',
  'similarity_to_ref': np.float64(0.9434776690209127)},
 {'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'OpenAI GPT OSS 120B',
  'run_id': 'P1_01_01',
  'similarity_to_ref': np.float64(0.9447895985244827)},
 {'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'Llama 3.1 SauerkrautLM 70B Instruct',
  'run_id': 'P1_01_01',
  'similarity_to_ref': np.float64(0.9131805176355671)},
 {'audio_id': 'Anam_sexueller_Missbrauch',
  'model_name': 'Llama 3.1 SauerkrautLM 70B Instruct',
  'run_id': 'P1_01_02',
  'similarity_to_ref': np.float64(0.9173964144278788)

In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

emb_col = "Alibaba-NLP/gte-multilingual-base"
results = []

for audio in df["audio_id"].unique():
    subset = df[df["audio_id"] == audio]
    ref = subset[subset["model_name"] == "OpenAI GPT-5"]
    if len(ref) != 1:
        continue  # skip if keine eindeutige Referenz
    
    ref_emb = np.array(ref.iloc[0][emb_col]).reshape(1, -1)
    others = subset[subset["model_name"] != "OpenAI GPT-5"]

    for _, row in others.iterrows():
        sim = cosine_similarity(ref_emb, np.array(row[emb_col]).reshape(1, -1))[0, 0]
        results.append({
            "audio_id": audio,
            "model_name": row["model_name"],
            "run_id": row["run_id"],
            "similarity_to_ref": sim
        })


import pandas as pd
res_df = pd.DataFrame(results)
model_means = res_df.groupby("model_name")["similarity_to_ref"].mean().sort_values(ascending=False)
print(model_means)



model_name
OpenAI GPT OSS 120B                    0.918454
MedGemma 27B Instruct                  0.890684
Llama 3.1 SauerkrautLM 70B Instruct    0.879675
Meta Llama 3.1 8B Instruct             0.857304
Name: similarity_to_ref, dtype: float64


In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

emb_col = "embedding_Llama-3.1_8B-Instruct"
results = []

for audio in df["audio_id"].unique():
    subset = df[df["audio_id"] == audio]
    ref = subset[subset["model_name"] == "OpenAI GPT-5"]
    if len(ref) != 1:
        continue  # skip if keine eindeutige Referenz
    
    ref_emb = np.array(ref.iloc[0][emb_col]).reshape(1, -1)
    others = subset[subset["model_name"] != "OpenAI GPT-5"]

    for _, row in others.iterrows():
        sim = cosine_similarity(ref_emb, np.array(row[emb_col]).reshape(1, -1))[0, 0]
        results.append({
            "audio_id": audio,
            "model_name": row["model_name"],
            "run_id": row["run_id"],
            "similarity_to_ref": sim
        })


import pandas as pd
res_df = pd.DataFrame(results)
model_means = res_df.groupby("model_name")["similarity_to_ref"].mean().sort_values(ascending=False)
print(model_means)



model_name
Llama 3.1 SauerkrautLM 70B Instruct    0.824693
OpenAI GPT OSS 120B                    0.816359
Meta Llama 3.1 8B Instruct             0.785653
MedGemma 27B Instruct                  0.729495
Name: similarity_to_ref, dtype: float64


In [12]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

emb_col = "BAAI/bge-m3"
results = []

for audio in df["audio_id"].unique():
    subset = df[df["audio_id"] == audio]
    ref = subset[subset["model_name"] == "OpenAI GPT-5"]
    if len(ref) != 1:
        continue  # skip if keine eindeutige Referenz
    
    ref_emb = np.array(ref.iloc[0][emb_col]).reshape(1, -1)
    others = subset[subset["model_name"] != "OpenAI GPT-5"]

    for _, row in others.iterrows():
        sim = cosine_similarity(ref_emb, np.array(row[emb_col]).reshape(1, -1))[0, 0]
        results.append({
            "audio_id": audio,
            "model_name": row["model_name"],
            "run_id": row["run_id"],
            "similarity_to_ref": sim
        })


import pandas as pd
res_df = pd.DataFrame(results)
model_means = res_df.groupby("model_name")["similarity_to_ref"].mean().sort_values(ascending=False)
print(model_means)



model_name
OpenAI GPT OSS 120B                    0.900191
MedGemma 27B Instruct                  0.880564
Llama 3.1 SauerkrautLM 70B Instruct    0.854236
Meta Llama 3.1 8B Instruct             0.847873
Name: similarity_to_ref, dtype: float64


In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

emb_col = "aari1995/German_Semantic_V3b"
results = []

for audio in df["audio_id"].unique():
    subset = df[df["audio_id"] == audio]
    ref = subset[subset["model_name"] == "OpenAI GPT-5"]
    if len(ref) != 1:
        continue  # skip if keine eindeutige Referenz
    
    ref_emb = np.array(ref.iloc[0][emb_col]).reshape(1, -1)
    others = subset[subset["model_name"] != "OpenAI GPT-5"]

    for _, row in others.iterrows():
        sim = cosine_similarity(ref_emb, np.array(row[emb_col]).reshape(1, -1))[0, 0]
        results.append({
            "audio_id": audio,
            "model_name": row["model_name"],
            "run_id": row["run_id"],
            "similarity_to_ref": sim
        })


import pandas as pd
res_df = pd.DataFrame(results)
model_means = res_df.groupby("model_name")["similarity_to_ref"].mean().sort_values(ascending=False)
print(model_means)



model_name
OpenAI GPT OSS 120B                    0.977391
MedGemma 27B Instruct                  0.966842
Llama 3.1 SauerkrautLM 70B Instruct    0.937614
Meta Llama 3.1 8B Instruct             0.921478
Name: similarity_to_ref, dtype: float64


## Konsistenzprüfung 

In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

emb_col = "embedding_Jina_v3"
consistency_results = []

# Iteriere über alle Modelle
for model in df["model_name"].unique():
    model_subset = df[df["model_name"] == model]

    audio_scores = []

    # Für jede Audio-ID dieses Modells
    for audio_id, subset in model_subset.groupby("audio_id"):
        if len(subset) < 2:
            continue  # braucht mindestens 2 Wiederholungen zum Vergleich

        # Alle Embeddings als Liste von Arrays
        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]

        # Alle möglichen Paare bilden
        pairs = list(itertools.combinations(embeddings, 2))

        # Kosinus-Ähnlichkeiten aller Paare berechnen
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        # Mittelwert der Similarities für dieses Audio
        mean_sim = np.mean(sims)
        audio_scores.append(mean_sim)

    # Mittelwert über alle Audios (Konsistenz dieses Modells)
    if len(audio_scores) > 0:
        model_mean_consistency = np.mean(audio_scores)
        consistency_results.append({
            "model_name": model,
            "mean_consistency": model_mean_consistency
        })

# In DataFrame umwandeln und sortieren
consistency_df = pd.DataFrame(consistency_results).sort_values("mean_consistency", ascending=False)
print(consistency_df)


                            model_name  mean_consistency
1                MedGemma 27B Instruct          0.976860
3                  OpenAI GPT OSS 120B          0.964204
0  Llama 3.1 SauerkrautLM 70B Instruct          0.963653
2           Meta Llama 3.1 8B Instruct          0.947726


In [30]:
subset

,experiment_id,audio_id,model_name,run_id,type,tags,text,embedding_Llama-3.1_8B-Instruct,embedding_Jina_v3,BAAI/bge-m3,Alibaba-NLP/gte-multilingual-base
183,1968,Anam_sexueller_Missbrauch,OpenAI GPT OSS 120B,P1_01_04,output,"[AI, LLM, STT, Summary, OpenAI GPT OSS 120B]","**Patient:** Marie Oppermann, geboren am 5. No...","[-0.007373909931629896, -0.01190749928355217, ...","[0.02685546875, -0.08984375, 0.05859375, 0.155...","[-0.04866880923509598, 0.023393869400024414, -...","[-0.0963086187839508, 0.04987655580043793, -0...."
184,1967,Anam_sexueller_Missbrauch,OpenAI GPT OSS 120B,P1_01_03,output,"[AI, LLM, STT, Summary, OpenAI GPT OSS 120B]","**Patient:** Marie Oppermann, geboren am 5. No...","[0.0007560037192888558, -0.011390907689929008,...","[0.0181884765625, -0.10302734375, 0.0732421875...","[-0.039278265088796616, 0.025669649243354797, ...","[-0.07454384863376617, 0.04515312984585762, -0..."
185,1966,Anam_sexueller_Missbrauch,OpenAI GPT OSS 120B,P1_01_02,output,"[AI, LLM, STT, Summary, OpenAI GPT OSS 120B]","**Patient:** Marie Oppermann, geboren am 5. No...","[0.002202144358307123, -0.011161179281771183, ...","[0.02099609375, -0.09423828125, 0.060302734375...","[-0.02879694476723671, 0.023191360756754875, -...","[-0.09194938093423843, 0.05312851443886757, -0..."
186,1965,Anam_sexueller_Missbrauch,OpenAI GPT OSS 120B,P1_01_01,output,"[AI, LLM, STT, Summary, OpenAI GPT OSS 120B]","**Patient:** Marie Oppermann, geboren am 5. No...","[-0.002898529404774308, -0.012305077165365219,...","[0.0208740234375, -0.1103515625, 0.07568359375...","[-0.04584423452615738, 0.029720488935709, -0.0...","[-0.09786396473646164, 0.04543650150299072, -0..."


In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

emb_col = "BAAI/bge-m3"
consistency_results = []

# Iteriere über alle Modelle
for model in df["model_name"].unique():
    model_subset = df[df["model_name"] == model]
    print(model)

    audio_scores = []

    # Für jede Audio-ID dieses Modells
    for audio_id, subset in model_subset.groupby("audio_id"):
        if len(subset) < 2:
            continue  # braucht mindestens 2 Wiederholungen zum Vergleich

        # Alle Embeddings als Liste von Arrays
        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]

        # Alle möglichen Paare bilden
        pairs = list(itertools.combinations(embeddings, 2))

        # Kosinus-Ähnlichkeiten aller Paare berechnen
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        # Mittelwert der Similarities für dieses Audio
        mean_sim = np.mean(sims)
        audio_scores.append(mean_sim)

    # Mittelwert über alle Audios (Konsistenz dieses Modells)
    if len(audio_scores) > 0:
        model_mean_consistency = np.mean(audio_scores)
        consistency_results.append({
            "model_name": model,
            "mean_consistency": model_mean_consistency
        })

# In DataFrame umwandeln und sortieren
consistency_df = pd.DataFrame(consistency_results).sort_values("mean_consistency", ascending=False)
print(consistency_df)


Llama 3.1 SauerkrautLM 70B Instruct
MedGemma 27B Instruct
Meta Llama 3.1 8B Instruct
OpenAI GPT OSS 120B
OpenAI GPT-5
GPT-5
                            model_name  mean_consistency
1                MedGemma 27B Instruct          0.930038
3                  OpenAI GPT OSS 120B          0.928566
0  Llama 3.1 SauerkrautLM 70B Instruct          0.925269
2           Meta Llama 3.1 8B Instruct          0.891178


In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

emb_col = "Alibaba-NLP/gte-multilingual-base"
consistency_results = []

# Iteriere über alle Modelle
for model in df["model_name"].unique():
    model_subset = df[df["model_name"] == model]

    audio_scores = []

    # Für jede Audio-ID dieses Modells
    for audio_id, subset in model_subset.groupby("audio_id"):
        if len(subset) < 2:
            continue  # braucht mindestens 2 Wiederholungen zum Vergleich

        # Alle Embeddings als Liste von Arrays
        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]

        # Alle möglichen Paare bilden
        pairs = list(itertools.combinations(embeddings, 2))

        # Kosinus-Ähnlichkeiten aller Paare berechnen
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        # Mittelwert der Similarities für dieses Audio
        mean_sim = np.mean(sims)
        audio_scores.append(mean_sim)

    # Mittelwert über alle Audios (Konsistenz dieses Modells)
    if len(audio_scores) > 0:
        model_mean_consistency = np.mean(audio_scores)
        consistency_results.append({
            "model_name": model,
            "mean_consistency": model_mean_consistency
        })

# In DataFrame umwandeln und sortieren
consistency_df = pd.DataFrame(consistency_results).sort_values("mean_consistency", ascending=False)
print(consistency_df)


                            model_name  mean_consistency
1                MedGemma 27B Instruct          0.975352
0  Llama 3.1 SauerkrautLM 70B Instruct          0.968779
3                  OpenAI GPT OSS 120B          0.958709
2           Meta Llama 3.1 8B Instruct          0.940579


In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

emb_col = "embedding_Llama-3.1_8B-Instruct"
consistency_results = []


# Zeilen mit fehlenden Embeddings entfernen (später brauchen wir das nicht mehr aber nur jetzt, weil ein Experient angepasst werden muss (none))
df = df.dropna(subset=[emb_col])


# Iteriere über alle Modelle
for model in df["model_name"].unique():
    model_subset = df[df["model_name"] == model]
    print(model)

    audio_scores = []

    # Für jede Audio-ID dieses Modells
    for audio_id, subset in model_subset.groupby("audio_id"):
        if len(subset) < 2:
            continue  # braucht mindestens 2 Wiederholungen zum Vergleich

        # Alle Embeddings als Liste von Arrays
        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]

        # Alle möglichen Paare bilden
        pairs = list(itertools.combinations(embeddings, 2))

        # Kosinus-Ähnlichkeiten aller Paare berechnen
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        # Mittelwert der Similarities für dieses Audio
        mean_sim = np.mean(sims)
        audio_scores.append(mean_sim)

    # Mittelwert über alle Audios (Konsistenz dieses Modells)
    if len(audio_scores) > 0:
        model_mean_consistency = np.mean(audio_scores)
        consistency_results.append({
            "model_name": model,
            "mean_consistency": model_mean_consistency
        })

# In DataFrame umwandeln und sortieren
consistency_df = pd.DataFrame(consistency_results).sort_values("mean_consistency", ascending=False)
print(consistency_df)


Llama 3.1 SauerkrautLM 70B Instruct
MedGemma 27B Instruct
Meta Llama 3.1 8B Instruct
OpenAI GPT OSS 120B
OpenAI GPT-5
GPT-5
                            model_name  mean_consistency
0  Llama 3.1 SauerkrautLM 70B Instruct          0.943093
3                  OpenAI GPT OSS 120B          0.911676
2           Meta Llama 3.1 8B Instruct          0.911231
1                MedGemma 27B Instruct          0.801991


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

emb_col = "aari1995/German_Semantic_V3b"
consistency_results = []


# Zeilen mit fehlenden Embeddings entfernen (später brauchen wir das nicht mehr aber nur jetzt, weil ein Experient angepasst werden muss (none))
df = df.dropna(subset=[emb_col])


# Iteriere über alle Modelle
for model in df["model_name"].unique():
    model_subset = df[df["model_name"] == model]
    print(model)

    audio_scores = []

    # Für jede Audio-ID dieses Modells
    for audio_id, subset in model_subset.groupby("audio_id"):
        if len(subset) < 2:
            continue  # braucht mindestens 2 Wiederholungen zum Vergleich

        # Alle Embeddings als Liste von Arrays
        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]

        # Alle möglichen Paare bilden
        pairs = list(itertools.combinations(embeddings, 2))

        # Kosinus-Ähnlichkeiten aller Paare berechnen
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        # Mittelwert der Similarities für dieses Audio
        mean_sim = np.mean(sims)
        audio_scores.append(mean_sim)

    # Mittelwert über alle Audios (Konsistenz dieses Modells)
    if len(audio_scores) > 0:
        model_mean_consistency = np.mean(audio_scores)
        consistency_results.append({
            "model_name": model,
            "mean_consistency": model_mean_consistency
        })

# In DataFrame umwandeln und sortieren
consistency_df = pd.DataFrame(consistency_results).sort_values("mean_consistency", ascending=False)
print(consistency_df)


Llama 3.1 SauerkrautLM 70B Instruct
MedGemma 27B Instruct
Meta Llama 3.1 8B Instruct
OpenAI GPT OSS 120B
OpenAI GPT-5
GPT-5
                            model_name  mean_consistency
3                  OpenAI GPT OSS 120B          0.981438
1                MedGemma 27B Instruct          0.979042
0  Llama 3.1 SauerkrautLM 70B Instruct          0.970370
2           Meta Llama 3.1 8B Instruct          0.942120


## Anzahl der Tokens in den Texten

In [8]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import pandas as pd

# ============================================================
# 1. Original-DF laden
# ============================================================
df = pd.read_parquet("last_clean_llm_experiments_with_embeddings.parquet")

# ============================================================
# 2. Tokenizer holen (Jina v3)
# ============================================================
st_model = SentenceTransformer(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True
)
tokenizer = st_model[0].tokenizer

# ============================================================
# 3. Neues DataFrame vorbereiten
# ============================================================
new_rows = []   # Hier speichern wir die Ergebnisse

# ============================================================
# 4. Token-Anzahl pro Text berechnen
# ============================================================
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Token zählen"):
    text = row["text"] if pd.notna(row["text"]) else ""

    # Tokenisieren (ungeschnitten, für echte Anzahl)
    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=True
    )
    token_count = tokens["input_ids"].shape[1]

    new_rows.append({
        "audio_id": row["audio_id"],
        "model_name": row["model_name"],
        "run_id": row["run_id"],
        "token_count": token_count,
        "char_length": len(text),
        "text": text
    })

# ============================================================
# 5. Neues DataFrame erstellen
# ============================================================
token_df = pd.DataFrame(new_rows)

# Optional: speichern
token_df.to_parquet("token_counts_jina_v3.parquet")

# ============================================================
# 6. Erste Zeilen ansehen
# ============================================================
print(token_df.head())


/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed.

             audio_id                           model_name    run_id  \
0  Anam_Beinschmerzen  Llama 3.1 SauerkrautLM 70B Instruct  P1_01_03   
1  Anam_Beinschmerzen  Llama 3.1 SauerkrautLM 70B Instruct  P1_01_01   
2  Anam_Beinschmerzen                MedGemma 27B Instruct  P1_01_01   
3  Anam_Beinschmerzen  Llama 3.1 SauerkrautLM 70B Instruct  P1_01_04   
4  Anam_Beinschmerzen                MedGemma 27B Instruct  P1_01_03   

   token_count  char_length                                               text  
0          446         1849  Patient: Frau Mayerhose, 55 Jahre alt, geboren...  
1          503         2030  Patient: Frau Mayerhose Katarina, 55 Jahre alt...  
2         2212         8063  Okay, hier ist die Zusammenfassung des Gespräc...  
3          559         2275  Patient: Frau Mayerhose Katarina, 55 Jahre alt...  
4         2817        11290  Okay, hier ist die Zusammenfassung des Gespräc...  


In [9]:
token_df[token_df["token_count"] > 4000]


,audio_id,model_name,run_id,token_count,char_length,text
65,Anam_Aengstliche-Patientin,MedGemma 27B Instruct,P1_01_04,14232,56620,"Okay, hier ist die Zusammenfassung des Transkr..."
87,Anam_Fahrradunfall_01,OpenAI GPT OSS 120B,P1_01_03,4024,14108,"**Patient:** Maria Brinkmann, geboren am 5. Ma..."
104,Anam_Covid-19_02,Meta Llama 3.1 8B Instruct,P1_01_02,4150,17694,"### Patient: Frau Doktor, ich bin Sebastian Fe..."
105,Anam_Covid-19_02,Meta Llama 3.1 8B Instruct,P1_01_01,4164,17733,"### Patient: Frau Doktor, ich bin Sebastian Fe..."
148,Anam_Pneumonie_02,MedGemma 27B Instruct,P1_01_01,4054,16006,"Okay, hier ist die Zusammenfassung des Gespräc..."
217,Anam_Fahrradunfall_02,Meta Llama 3.1 8B Instruct,P1_01_01,7675,29062,"### Patient: Frau Julia Becken-Westphalen, 33 ..."
218,Anam_Fahrradunfall_02,Meta Llama 3.1 8B Instruct,P1_01_02,4667,18116,"### Patient: Frau Julia Becken-Westphalen, 33 ..."
396,Anam_Fremdanamnese,Meta Llama 3.1 8B Instruct,P1_01_01,4004,16041,"### Patient: Frau Weiß, 83 Jahre alt, ist die ..."


In [10]:
import pandas as pd

df = pd.read_parquet("token_counts_jina_v3.parquet")


In [11]:
import pandas as pd

# -----------------------------
# 1. Parquet laden
# -----------------------------
df = pd.read_parquet("token_counts_jina_v3.parquet")

# -----------------------------
# 2. Modelle ausschließen
# -----------------------------
exclude_models = ["OpenAI GPT-5", "GPT-5"]

df_filtered = df[~df["model_name"].isin(exclude_models)]

# Optional: Check
print("Verbleibende Modelle:")
print(df_filtered["model_name"].unique())

# -----------------------------
# 3. Mittelwert der Token pro Modell
# -----------------------------
mean_tokens = (
    df_filtered
    .groupby("model_name", as_index=False)["token_count"]
    .mean()
    .rename(columns={"token_count": "mean_token_count"})
    .sort_values("mean_token_count", ascending=False)
)

print("\nMittelwert der Token pro Modell (gefiltert):")
print(mean_tokens)


Verbleibende Modelle:
['Llama 3.1 SauerkrautLM 70B Instruct' 'MedGemma 27B Instruct'
 'Meta Llama 3.1 8B Instruct' 'OpenAI GPT OSS 120B']

Mittelwert der Token pro Modell (gefiltert):
                            model_name  mean_token_count
1                MedGemma 27B Instruct       2027.514563
3                  OpenAI GPT OSS 120B       1923.759615
2           Meta Llama 3.1 8B Instruct        935.721154
0  Llama 3.1 SauerkrautLM 70B Instruct        545.180952
